In [1]:
import numpy as np
import sys, pathlib, os
sys.path.append(str(pathlib.Path(os.path.abspath('')) / ".."))

import NuLattice.HF.hartree_fock as hf
import NuLattice.lattice as lat
import NuLattice.operators.two_body_operators as twbo
import NuLattice.operators.one_body_operators as obo
import NuLattice.operators.three_body_operators as thbo
import NuLattice.constants_NLEFT as nleftConsts

In [ ]:
myL= 4

# 2017 paper, arXiv:1702.05177v2
bpi = 0.7
a = 1.0 / 100.0
sL = 0.08
sNL = 0.08
c0 = -0.185 / a
hbarc = nleftConsts.hbarc  # 197 MeV fm: a = (100 MeV)^-1 = 1.97 fm as in the paper

# Uncomment the following lines to enable the Hamiltonian
# from the 2018 paper
# # 2018 paper, arXiv:1812.10928v2
# a  = 1.0/150.0
# sNL= 0.5
# sL = 0.061
# c0 = -3.41e-7/a**3
# c3 = -1.4e-14/a**6
# # the paper quotes a = (150 MeV)^-1 = 1.32 fm, which needs the CODATA hbar*c
# # (1.3155 fm) instead of nleftConsts.hbarc = 197 MeV fm (1.3133 fm)
# import NuLattice.constants as consts
# hbarc = consts.hbarc  # 197.3269804 MeV fm


lattice = lat.get_lattice(myL)
nsite = len(lattice)
print("number of lattice sites =", nsite)

my_basis = lat.get_sp_basis(myL)
nstat =  len(my_basis)
print("number of single-particle states =", nstat)

number of lattice sites = 64
number of single-particle states = 256


In [3]:
# case = "neutron_matter"
case = "nuclear_matter"

In [4]:
tKin=obo.tKin(myL, 3, a,mass=nleftConsts.mass)
# two-body just on one site; need to multiply strength by L**3
V2sparse = twbo.shortRangeV_2body(lattice, myL, sL, sNL, c0*myL**3, spin = 2, isospin = 2, verbose = True, sites=[[2,2,2]]) 

data_type = V2sparse.dtype
print("data type of interaction", data_type)
my_cont2=twbo.sparse_to_list_2body(V2sparse, myL)
del V2sparse

Generating Densities...Done
Performing Local Smearing...Done
Generating Interaction...Interaction Generated
data type of interaction float64


In [5]:
W3sparse = None
# Uncomment the following lines to enable the 3-body force
# # three-body just on one site; need to multiply strength by L**3
# W3sparse = thbo.shortRangeV_3body(lattice, myL, sL, sNL, c3*myL**3, spin = 2, isospin = 2, verbose = True,max_mem=1e9, sites=[[2,2,2]])
# data_type = W3sparse.dtype
# print("data type of interaction", data_type)

In [6]:
# For nuclear matter:
# Diagonalize kinetic energy and make density matrix

Tkin_mat = hf.get_1body_matrix(tKin,nstat,dtype=data_type)
vals, vecs = np.linalg.eigh(Tkin_mat)

In [7]:
if case == "nuclear_matter":
    Tkin_mat = hf.get_1body_matrix(tKin,nstat,dtype=data_type)
elif case == "neutron_matter":
    # we need to separate neutrons and protons
    tauz = obo.tau_z(lattice, myL)
    sep_fac=10.0**5
    neut_Tkin = obo.list_to_sparse1b(tKin) - sep_fac*obo.list_to_sparse1b(tauz) # to push protons up and neutrons down
    neut_Tkin_list=obo.sparse1b_to_list(neut_Tkin)
    Tkin_mat = hf.get_1body_matrix(neut_Tkin_list,nstat,dtype=data_type)
else: 
    print("WRONG case ARGUMENT: valid choices are 'nuclear_matter' or 'neutron_matter' ")

vals, vecs = np.linalg.eigh(Tkin_mat)

In [8]:
# compute magic numbers / shell closures of kinetic energy
magic_nums=[]
for i in range(1,nstat):
    if vals[i]-vals[i-1] > 1.e-4:
        magic_nums.append(i)

magic_nums.append(nstat)
print(magic_nums)

[4, 28, 76, 88, 120, 168, 216, 228, 252, 256]


In [9]:
print("A, rho, E/A")
a_lat = a * hbarc  # lattice spacing in fm
selected_nums=magic_nums
for aa in selected_nums:
    dens=hf.contract("pi,qi->pq", vecs[:,0:aa], np.conjugate(vecs[:,0:aa]) )
    if W3sparse is not None:
        erg=hf.HF_energy(tKin, my_cont2, W3sparse, dens, w3_sparse=True)
    else:
        erg=hf.HF_energy(tKin, my_cont2, [], dens, w3_sparse=False)
    num = np.real(np.trace(dens))
    print(f'{num},{num/(myL*a_lat)**3},{np.real(erg)/num}')

A, rho, E/A
4.000000000000005,0.008174879106617832,-4.556730102383997
28.000000000000004,0.05722415374632475,-12.649676625637346
76.0,0.15532270302573858,-26.753419022822264
88.00000000000001,0.17984734034559208,-31.08775359523844
120.00000000000003,0.24524637319853468,-34.404766400103476
168.0,0.34334492247794846,-41.57429194846729
216.0,0.4414434717573623,-39.63399600864971
228.0,0.46596810907721575,-39.373654614088174
252.00000000000003,0.5150173837169227,-34.728079089467755
256.0,0.5231922628235405,-33.31420109464028
